In [ ]:
python
#!/usr/bin/env python
# inference_pipeline_with_custom_transformer.py

import torch
import os
import pickle
import requests
import re
import time
import torch.nn as nn
from transformers import AutoModel, AutoTokenizer, AutoModelForCausalLM

# ─── 1. Configuration ────────────────────────────────────────────────────────
DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ARTIFACT_DIR = "/Users/dhyeyk/Desktop/UIUC/Dr.Bot_Challenge/Model_Code/non_verbalized_saved_model"
MAX_LEN      = 64
WIKI_API     = "https://en.wikipedia.org/w/api.php"
SECTIONS     = [
    "Signs and symptoms",
    "Causes",
    "Diagnosis",
    "Prevention",
    "Treatment",
]

# ─── 2. Reload PubMedBERT Classifier ────────────────────────────────────────
tok = AutoTokenizer.from_pretrained(ARTIFACT_DIR)

label_embs = torch.load(
    os.path.join(ARTIFACT_DIR, "label_embs.pt"),
    map_location=DEVICE
).to(DEVICE)
with open(os.path.join(ARTIFACT_DIR, "id2label.pkl"), "rb") as f:
    id2label = pickle.load(f)

bert = AutoModel.from_pretrained(
    "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract"
).to(DEVICE).eval()

class LabelEmbCls(nn.Module):
    def __init__(self, base, lbl_emb):
        super().__init__()
        self.bert = base
        self.lbl_E = nn.Parameter(lbl_emb, requires_grad=False)
        self.tau   = nn.Parameter(torch.tensor(1.0))

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        cls = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids
        ).last_hidden_state[:, 0]
        return torch.matmul(cls, self.lbl_E.T) / self.tau

model = LabelEmbCls(bert, label_embs).to(DEVICE)
state = torch.load(
    os.path.join(ARTIFACT_DIR, "classifier.pt"),
    map_location=DEVICE
)
model.load_state_dict(state)
model.eval()

# ─── 3. Load your fine-tuned physician-style transformer ────────────────────
PHYS_MODEL_DIR = "./physician_transformer"
phys_tok       = AutoTokenizer.from_pretrained(PHYS_MODEL_DIR)
phys_model     = AutoModelForCausalLM.from_pretrained(
    PHYS_MODEL_DIR,
    torch_dtype=torch.float16
).to(DEVICE).eval()

# ─── Patch tokenizer/model for EOS, PAD, and left-side truncation ────────
if phys_tok.eos_token_id is None:
    # add EOS if missing
    phys_tok.add_special_tokens({"eos_token": ""})
    phys_model.resize_token_embeddings(len(phys_tok))
phys_model.config.eos_token_id = phys_tok.eos_token_id
phys_model.config.pad_token_id = phys_tok.eos_token_id
phys_tok.truncation_side = "left"

# ─── 4. Wikipedia Helpers ────────────────────────────────────────────────────
session = requests.Session()
session.headers.update({"User-Agent": "DrBot/1.0 (youremail@domain.edu)"})

def mw_request(params):
    r = session.get(WIKI_API, params=params, timeout=10)
    r.raise_for_status()
    return r.json()

def resolve_topic(topic: str) -> str:
    data = mw_request({
        "action":   "query",
        "list":     "search",
        "srsearch": topic,
        "srlimit":  1,
        "format":   "json"
    })
    hits = data.get("query", {}).get("search", [])
    return hits[0]["title"] if hits else topic.title()

def clean_text(html: str) -> str:
    text = re.sub(r"<!--.*?-->", "", html, flags=re.DOTALL)
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"\[\d+\]", "", text)
    text = re.sub(r"\[edit\]", "", text)
    text = re.sub(r"\{\{.*?\}\}", "", text)
    return re.sub(r"\s+", " ", text).strip()

def fetch_section_text(topic: str, section_title: str) -> str:
    title = resolve_topic(topic)
    secs  = mw_request({
        "action":    "parse",
        "page":      title,
        "prop":      "sections",
        "redirects": True,
        "format":    "json"
    }).get("parse", {}).get("sections", [])
    idx = next((s["index"] for s in secs if s["line"].lower() == section_title.lower()), None)
    if not idx:
        return ""
    html = mw_request({
        "action":    "parse",
        "page":      title,
        "prop":      "text",
        "section":   idx,
        "redirects": True,
        "format":    "json"
    })["parse"]["text"]["*"]
    return clean_text(html)

def fetch_intro(topic: str, char_limit: int = 2000) -> str:
    title = resolve_topic(topic)
    pages = mw_request({
        "action":      "query",
        "titles":      title,
        "prop":        "extracts",
        "exintro":     True,
        "explaintext": True,
        "redirects":   True,
        "format":      "json"
    }).get("query", {}).get("pages", {})
    return clean_text(next(iter(pages.values())).get("extract", ""))[:char_limit]

# ─── 5. Inference Loop ───────────────────────────────────────────────────────
print("Model loaded!  Enter queries (type 'exit' or 'q' to quit)\n")

while True:
    q = input("Query ▶ ").strip()
    if q.lower() in ("exit", "quit", "q"):
        break

    # 5a) PubMedBERT → raw_topic
    enc = tok(
        "SeverityNormal -- " + q,
        truncation=True,
        max_length=MAX_LEN,
        padding="max_length",
        return_tensors="pt"
    ).to(DEVICE)
    with torch.no_grad():
        raw_topic = id2label[torch.argmax(model(**enc), dim=-1).item()]

    # 5b) Resolve & fetch Wiki
    topic = resolve_topic(raw_topic)
    print(f"\n→ Using topic for lookup: {topic}\n")

    intro = fetch_intro(topic)
    print("--- BACKGROUND ---\n", intro, "\n")
    context = intro + "\n"

    for sec in SECTIONS:
        sec_txt = fetch_section_text(topic, sec)
        if not sec_txt:
            continue
        print(f"--- {sec.upper()} ---\n{sec_txt}\n")
        context += f"{sec}:\n{sec_txt}\n"
        time.sleep(0.2)

    # 5c) Build prompt & generate
    prompt = (
        "You are a board-certified physician. "
        "Answer the question based on the provided information "
        "in a concise, empathetic, and clinically accurate style.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {q}\nAnswer:\n"
    )
    inputs = phys_tok(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024,
        padding="max_length"
    ).to(DEVICE)

    with torch.no_grad():
        outputs = phys_model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=phys_tok.pad_token_id,
            eos_token_id=phys_tok.eos_token_id,
        )

    # 5d) Decode & post-process
    raw = phys_tok.decode(outputs[0], skip_special_tokens=False)
    # strip everything after the first EOS
    eos_token = phys_tok.eos_token or ""
    raw = raw.split(eos_token, 1)[0]
    # extract only the answer part
    answer = raw.split("Answer:")[-1].strip()

    print("\n--- RESPONSE ---\n", answer, "\n")
    print("=" * 80 + "\n")